In [5]:
from pathlib import Path
import pyodbc
import toml
import pandas as pd

config_path = Path.cwd().parents[0] / "config" / "config.toml"
cfg = toml.load(config_path)

db = cfg["database"]

conn_str = (
    f"DRIVER={{{db['driver']}}};"
    f"SERVER={db['server']};"
    f"DATABASE={db['database']};"
    f"Trusted_Connection={db['trusted_connection']};"
    f"TrustServerCertificate=yes;"
)

conn = pyodbc.connect(conn_str)

In [6]:
import pandas as pd

query = """
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN Revenue IS NULL THEN 1 ELSE 0 END) AS null_revenue_rows,
    COUNT(DISTINCT SalesOrderID) AS distinct_orders,
    COUNT(DISTINCT SalesOrderDetailID) AS distinct_order_lines,
    COUNT(*) - COUNT(DISTINCT SalesOrderDetailID) AS duplicate_line_rows,
    SUM(Revenue) AS total_revenue,
    COUNT(DISTINCT CustomerID) AS customer_count,
    SUM(Revenue) / COUNT(DISTINCT SalesOrderID) AS AOV
FROM dbo.vw_fact_sales;
"""

validation_df = pd.read_sql(query, conn)

df = validation_df

if df["total_rows"][0] == 0:
    raise Exception("❌ No data in fact table")

if df["null_revenue_rows"][0] > 0:
    raise Exception("❌ Revenue contains NULLs")

if df["duplicate_line_rows"][0] > 0:
    raise Exception("❌ Duplicate order lines detected")

print("✅ Validation passed")
print(df)

C:\Users\hisuk\AppData\Local\Temp\ipykernel_59108\2708703791.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  validation_df = pd.read_sql(query, conn)


✅ Validation passed
   total_rows  null_revenue_rows  distinct_orders  distinct_order_lines  \
0      121317                  0            31465                121317   

   duplicate_line_rows  total_revenue  customer_count          AOV  
0                    0   1.098464e+08           19119  3491.065672  
